In [23]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

fatal: destination path 'kcw-analytics' already exists and is not an empty directory.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using folder: /content/drive/Shareddrives


In [24]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [25]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "MOBILE": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3012, 49)
Loaded: raw_syp_simas_sales_bills.csv -> (13202, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (28103, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38554, 38)
Loaded: raw_hq_icmas_products.csv -> (114984, 94)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154142, 41)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50262, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733804, 38)
Loaded: raw_hq_apmas_payable.csv -> (979, 20)
Loaded: raw_hq_armas_receivable.csv -> (2233, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276222, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13893, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13893, 32)


In [15]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/online/lazada"

In [18]:
import pandas as pd
import glob
import os

files = glob.glob(os.path.join(folder, "*.xlsx"))

df_lazada_raw = pd.concat(
    [
        pd.read_excel(
            f,
            sheet_name="Income Overview",
            header=0,        # explicitly say header is first row
            dtype={
                "หมายเลขคำสั่งซื้อ": "string",
                "SKU ร้านค้า": "string",
            }
        )
        for f in files
    ],
    ignore_index=True
)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [19]:
df_lazada_raw.columns

Index(['ระยะเวลาใบแจ้งยอด', 'รหัสรอบบิล', 'วันที่ทำรายการ',
       'ชื่อรายการธุรกรรม', 'จำนวนเงิน(รวมภาษี)', 'VAT Amount',
       'สถานะการโอนเงิน', 'วันที่ปรับปรุงเข้ายอดของฉัน', 'ความคิดเห็น',
       'วันที่สร้างคำสั่งซื้อ', 'หมายเลขคำสั่งซื้อ', 'รหัสสินค้าในคำสั่งซื้อ',
       'SKU ร้านค้า', 'Lazada SKU', 'WHT Amount', 'WHT รวมอยู่ในจำนวนเงินแล้ว',
       'สถานะคำสั่งซื้อ', 'ชื่อสินค้า', 'Short Code'],
      dtype='object')

In [20]:
df = df_lazada_raw.copy()

# ensure numeric
df["จำนวนเงิน(รวมภาษี)"] = (
    pd.to_numeric(df["จำนวนเงิน(รวมภาษี)"], errors="coerce")
)

# composite key
df["sku_txn_key"] = (
    df["SKU ร้านค้า"].astype("string") + "|" +
    df["หมายเลขคำสั่งซื้อ"].astype("string")
)

# aggregation
df_grouped = (
    df
    .groupby(["SKU ร้านค้า", "หมายเลขคำสั่งซื้อ"], dropna=False)
    .agg(
        amount_signed = ("จำนวนเงิน(รวมภาษี)", "sum"),
        amount_positive = ("จำนวนเงิน(รวมภาษี)", lambda s: s[s > 0].sum()),
        amount_negative = ("จำนวนเงิน(รวมภาษี)", lambda s: s[s < 0].sum()),
        line_count = ("จำนวนเงิน(รวมภาษี)", "size")
    )
    .reset_index()
)

In [21]:
df_grouped

,SKU ร้านค้า,หมายเลขคำสั่งซื้อ,amount_signed,amount_positive,amount_negative,line_count
0,07010524,1085363326036892,123.08,145.0,-21.92,4
1,08010644,1085698553012609,3208.74,3790.0,-581.26,5
2,08017381,1081633748789005,532.67,650.0,-117.33,5
3,08054588,1075422245572155,367.32,450.0,-82.68,5
4,08056195,1077981662469770,1212.64,1480.0,-267.36,5
...,...,...,...,...,...,...
332,35050150,1084565350116011,273.64,325.0,-51.36,4
333,35050222,1078444494538885,534.80,630.0,-95.20,10
334,35050251+13052133+13052135,1086834138416680,314.76,375.0,-60.24,4
335,5968962800-1764129397314-0,1084478547890365,1212.63,1480.0,-267.37,5


In [26]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()

In [33]:
df_left = df_grouped.copy()

# --- trim end ---
df_left["REF_TRIM_END"] = (
    df_left["หมายเลขคำสั่งซื้อ"]
    .astype("string")
    .str.strip()
    .str[:-1]
)

# --- trim front ---
df_left["REF_TRIM_FRONT"] = (
    df_left["หมายเลขคำสั่งซื้อ"]
    .astype("string")
    .str.strip()
    .str[1:]
)

# --- lookup table ---
df_lookup = (
    df_simas[["PO", "BILLNO"]]
    .dropna(subset=["PO"])
    .drop_duplicates()
    .copy()
)

df_lookup["PO"] = df_lookup["PO"].astype("string").str.strip()
df_lookup["BILLNO"] = df_lookup["BILLNO"].astype("string").str.strip()

# ========= 1️⃣ first merge (trim end) =========
df_out = df_left.merge(
    df_lookup,
    left_on="REF_TRIM_END",
    right_on="PO",
    how="left"
)

# ========= 2️⃣ second merge ONLY for NA =========
df_front = df_left.merge(
    df_lookup,
    left_on="REF_TRIM_FRONT",
    right_on="PO",
    how="left",
    suffixes=("", "_FRONT")
)

# ========= 3️⃣ fill missing =========
df_out["BILLNO"] = df_out["BILLNO"].fillna(df_front["BILLNO"])

In [34]:
df_end = df_left.merge(df_lookup, left_on="REF_TRIM_END", right_on="PO", how="left")
df_front = df_left.merge(df_lookup, left_on="REF_TRIM_FRONT", right_on="PO", how="left")

df_out = df_end.copy()
df_out["BILLNO"] = df_end["BILLNO"].combine_first(df_front["BILLNO"])

In [38]:
import numpy as np

# ---- count missing first (for audit) ----
missing_billno_cnt = df_out["BILLNO"].isna().sum()

# ---- build output table ----
df_final_pct = (
    df_out
    .dropna(subset=["BILLNO"])
    .copy()
)

df_final_pct["deduction_pct"] = np.where(
    df_final_pct["amount_positive"] > 0,
    100 * (1 - df_final_pct["amount_signed"] / df_final_pct["amount_positive"]),
    np.nan
)

df_final_pct["deduction_pct"] = df_final_pct["deduction_pct"].round(2)

df_final_pct = df_final_pct[["BILLNO", "deduction_pct"]]

In [40]:
print("Missing BILLNO rows:", missing_billno_cnt)

Missing BILLNO rows: 14


In [39]:
df_final_pct

,BILLNO,deduction_pct
0,TAD6902-488,15.12
1,TAD6902-518,15.34
2,TAD6902-115,18.05
3,TAD6902-295,18.37
4,TAD6902-552,18.06
...,...,...
335,TAD6902-151,15.38
336,TAD6902-413,15.80
337,TAD6902-598,15.11
338,TAD6903-023,16.06


In [41]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "03_curated",
    f"lazada.csv"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [42]:
df_final_pct.to_csv(folder, index=False, encoding="utf-8-sig")